In [ ]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
from scipy.integrate import simpson
from scipy.optimize import least_squares
import pandas as pd

def SIR(y, t, alpha, gamma):
    """
    :param y: initial condition
    :param t: time
    :param beta: para
    :param gamma: para
    :param S0: para
    :return: the SIR model ODEs
    """
    S, I, R = y
    dS = - alpha * S * I
    dI = alpha * S * I - gamma * I
    dR = gamma * I
    return [dS, dI, dR]

# initial condition
S0 = 9999
I0 = 1
R0 = 0
y = S0, I0, R0

# true parameters
alpha = 0.00003
gamma = 0.05
scales = [0.00001, 0.01, 10000]

# time
t = np.linspace(0, 500, 1000)

# simulate SIR
solution = scipy.integrate.odeint(SIR, [S0, I0, R0], t, args=(alpha, gamma))
S, I, R = solution.T


def residual(paras, I_data, t):
    """
    alpha = beta / N
    gamma = gamma
    S0 = S0
    :param paras:
    :param I_data:
    :param t:
    :return:
    """
    alpha, gamma, S0 = paras

    I0 = I_data[0]
    I_int = np.array([simpson(I_data[:i + 1], t[:i + 1]) for i in range(len(t))])
    I_int2 = np.array([simpson(I_data[:i + 1] ** 2, t[:i + 1]) for i in range(len(t))])
    int_double = 1 / 2 * (I_int ** 2)

    I_hat = I0 + (alpha * (S0 + I0) - gamma) * I_int - alpha * I_int2 - alpha * gamma * int_double
    return I_hat - I_data


# initial guess value of beta and gamma
alpha0 = 0.00001
gamma0 = 0.01
S0_initial = 6000
x0 = [alpha0, gamma0, S0_initial]

def evaluate(method, num_of_iteration, bounds=([0, 0, 0], [np.inf, np.inf, np.inf]), plot=False):
    print(f"--- Method : {method} ---")
    print(f"bounds {bounds}")

    estimated_alpha_list = []
    estimated_gamma_list = []
    estimated_S0_list = []

    nfev_list = []
    njev_list = []

    plot_alpha = []
    plot_gamma = []
    plot_S0 = []

    total_iterations = 0

    for _ in range(num_of_iteration):
        I_data = I + np.random.normal(0, 0.01 * I, size=I.shape)

        alpha_list = []
        gamma_list = []
        S0_list = []

        def callback(x, *args, **kwargs):
            alpha_list.append(x[0])
            gamma_list.append(x[1])
            S0_list.append(x[2])

        if method == 'lm':
            res = least_squares(residual, x0, args=(I_data, t), method=method)
        else:
            if bounds is not None:
                res = least_squares(
                    residual, x0, args=(I_data, t),
                    method=method, x_scale=scales, callback=callback, bounds=bounds
                )
            else:
                res = least_squares(
                    residual, x0, args=(I_data, t),
                    method=method, x_scale=scales, callback=callback
                )

        estimated_alpha_list.append(res.x[0])
        estimated_gamma_list.append(res.x[1])
        estimated_S0_list.append(res.x[2])

        nfev_list.append(res.nfev)
        njev_list.append(res.njev if res.njev is not None else np.nan)
        total_iterations += len(alpha_list)

        plot_alpha.append(alpha_list)
        plot_gamma.append(gamma_list)
        plot_S0.append(S0_list)

    estimated_alpha = np.mean(estimated_alpha_list)
    estimated_gamma = np.mean(estimated_gamma_list)
    estimated_S0 = np.mean(estimated_S0_list)

    print(f"Average estimated alpha among {num_of_iteration}: {estimated_alpha} --- real alpha {alpha}"
          f"\nAverage estimated gamma among {num_of_iteration}: {estimated_gamma} --- real gamma {gamma}"
          f"\nAverage estimated S0 among {num_of_iteration}: {estimated_S0} --- real S0 {S0}"
          f"\nAlpha error {abs((estimated_alpha - alpha) / alpha) * 100}"
          f"\nGamma error {abs((estimated_gamma - gamma) / gamma) * 100}"
          f"\nS0 error {abs((estimated_S0 - S0) / S0) * 100}")

    # print("Iterations:", total_iterations / num_of_iteration)
    print("Average nfev:", np.mean(nfev_list))
    print("Average njev:", np.nanmean(njev_list))
    print("status:", res.status)
    print("message:", res.message)

    if plot and method != 'lm':
        for i, alpha_hist in enumerate(plot_alpha):
            plt.plot(alpha_hist, label=f"alpha_{i}", marker='o')
        plt.axhline(y=alpha, label="True alpha", linestyle='-')
        plt.legend()
        plt.show()

        for i, gamma_hist in enumerate(plot_gamma):
            plt.plot(gamma_hist, label=f"gamma_{i}", marker='o')
        plt.axhline(y=gamma, label="True gamma", linestyle='-')
        plt.legend()
        plt.show()

        for i, S0_hist in enumerate(plot_S0):
            plt.plot(S0_hist, label=f"S0_{i}", marker='o')
        plt.axhline(y=S0, label="True S0", linestyle='-')
        plt.legend()
        plt.show()


In [ ]:
evaluate('trf', 20)

In [ ]:
evaluate('dogbox', 20)

In [ ]:
evaluate('lm', 20)

In [ ]:
evaluate('trf', 20, bounds=([0, 0, 0], [np.inf, np.inf, np.inf]))

In [ ]:
evaluate('dogbox', 20, bounds=([0, 0, 0], [np.inf, np.inf, np.inf]))

In [ ]:
evaluate('lm', 10, bounds=([0, 0, 0], [np.inf, np.inf, np.inf]))

In [ ]:
evaluate('trf', 50, plot=False)
evaluate('trf', 50, bounds=([0, 0, 0], [np.inf, np.inf, np.inf]), plot=False)
evaluate('dogbox', 50, plot=False)
evaluate('dogbox', 50, bounds=([0, 0, 0], [np.inf, np.inf, np.inf]), plot=False)
evaluate('lm', 50, plot=False)